![](https://www.soyhenry.com/_next/static/media/HenryLogo.bb57fd6f.svg)

# Cap 08 — Sistema Multi-Agente

profesor [Carlos Daniel Jiménez](danieljimenez88m@gmail.com)


Integramos todo en el Cultural Intelligence System completo:
- Sub-grafos compilados como nodos del grafo master
- Importar desde `src/cinematic_intelligence/`
- Streaming en tiempo real con `graph.stream()`
- Visualización con `xray=1`
- Preguntas cross-domain

**Dominio**: Los 3 dominios integrados  
**Flujo**: `CulturalRouter → [NolanGraph|KingGraph|DavisGraph] → Synthesizer → END`

In [ ]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

# Añadir src/ al path para importar cinematic_intelligence
src_path = Path("../src").resolve()
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

load_dotenv()
assert os.getenv("OPENAI_API_KEY"), "Falta OPENAI_API_KEY"
print(f"src/ en path: {src_path} ✓")

In [ ]:
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import MemorySaver

from cinematic_intelligence.graph import make_cultural_graph
from cinematic_intelligence.models import DomainEnum

print("cinematic_intelligence importado correctamente ✓")

## 1. Arquitectura del Sistema

```
CulturalState (MessagesState)
      │
      ▼
 node_router        ← LLM clasifica domain con structured output
      │
 cultural_route()   ← función pura lee state["domain"]
      │
 ┌────┴─────────────────┬──────────────────┐
 ▼                      ▼                  ▼
nolan_specialist   king_specialist   davis_specialist
 └──────────────────────┴──────────────────┘
                         │
                         ▼
                   node_synthesizer   ← CulturalResponse
                         │
                        END
```

In [ ]:
# Crear el grafo con MemorySaver para conversaciones persistentes
checkpointer = MemorySaver()
graph = make_cultural_graph(checkpointer=checkpointer)

print("Cultural Intelligence Graph compilado ✓")

## 2. Visualización con xray=1

`xray=1` muestra los internos de los sub-grafos.

In [ ]:
try:
    from IPython.display import Image, display
    # xray=1 para ver el grafo completo con internos
    png = graph.get_graph(xray=1).draw_mermaid_png()
    display(Image(png))
except Exception as e:
    print(f"PNG no disponible ({e}), mostrando Mermaid:")
    print(graph.get_graph().draw_mermaid())

## 3. Primera Consulta: Dominio Nolan

In [ ]:
config_1 = {"configurable": {"thread_id": "demo_nolan"}}

result = graph.invoke(
    {"messages": [HumanMessage(content="¿Cuál es la técnica narrativa más innovadora de Christopher Nolan?")]},
    config=config_1
)

print("=== Respuesta ===")
final = result.get("final_response")
if final:
    print(f"Dominio: {final.domain}")
    print(f"Confianza: {final.confidence}")
    print(f"\n{final.final_answer}")

## 4. Streaming en Tiempo Real

In [ ]:
config_2 = {"configurable": {"thread_id": "demo_streaming"}}

print("=== Streaming en tiempo real ===\n")
for chunk in graph.stream(
    {"messages": [HumanMessage(content="¿Cuáles son los temas del horror en Stephen King?")]},
    config=config_2,
    stream_mode="updates"
):
    node_name = list(chunk.keys())[0]
    print(f"[{node_name}] ", end="")
    # Mostrar campo relevante del chunk
    data = chunk[node_name]
    if "domain" in data:
        print(f"domain={data['domain']}")
    elif "domain_result" in data and data["domain_result"]:
        print(f"analysis preview: {str(data['domain_result'])[:80]}...")
    elif "final_response" in data and data["final_response"]:
        resp = data["final_response"]
        print(f"final_answer: {resp.final_answer[:100]}...")
    else:
        print(f"{list(data.keys())}")

## 5. Pregunta Cross-Domain

El sistema puede manejar preguntas que tocan múltiples dominios.

In [ ]:
config_3 = {"configurable": {"thread_id": "demo_cross_domain"}}

cross_query = """Tanto Nolan (Interstellar) como Miles Davis (Kind of Blue) 
tienen una obsesión por el tiempo y el espacio. ¿Cómo se manifiesta esto en cada uno?"""

result3 = graph.invoke(
    {"messages": [HumanMessage(content=cross_query)]},
    config=config_3
)

final3 = result3.get("final_response")
print(f"Dominio elegido: {final3.domain if final3 else 'N/A'}")
print(f"\n{final3.final_answer if final3 else 'No response'}")

## 6. Conversación Multi-Turn con Memoria

In [ ]:
config_4 = {"configurable": {"thread_id": "demo_multiturn"}}

turns = [
    "¿Cuál es el período más revolucionario de Miles Davis?",
    "¿Cómo influyó ese período en el jazz posterior?",
    "¿Y hay alguna conexión temática con las técnicas narrativas de Nolan?",
]

for i, query in enumerate(turns, 1):
    result = graph.invoke(
        {"messages": [HumanMessage(content=query)]},
        config=config_4
    )
    final = result.get("final_response")
    print(f"\n--- Turno {i} ---")
    print(f"Q: {query}")
    if final:
        print(f"A: {final.final_answer[:300]}...")

## Resumen del Módulo

Has completado el módulo de LangGraph. Aquí tienes una visión general de lo aprendido:

| Capítulo | Concepto clave | Dominio |
|----------|---------------|--------|
| 01 | TypedDict, StateGraph, START/END | Nolan |
| 02 | MessagesState, add_messages, LLM node | King |
| 03 | @tool, ToolNode, ReAct loop | Davis |
| 04 | add_conditional_edges, routing strategies | Los 3 |
| 05 | with_structured_output, Pydantic, fallback | Nolan |
| 06 | MemorySaver, thread_id, checkpointing | King |
| 07 | Send(), fan-out, parallel benchmark | Davis |
| 08 | Sub-graphs, streaming, multi-agent system | Los 3 |

### Próximos pasos

1. **Producción**: Reemplazar `MemorySaver` con `langgraph-checkpoint-postgres`
2. **Observabilidad**: Conectar LangSmith con `LANGCHAIN_TRACING_V2=true`
3. **Escalabilidad**: Añadir LangGraph Platform para deployment
4. **Evaluación**: Implementar métricas de calidad de respuesta